This notebook incrementally calculates profit estimates of performing skill based actions on an hourly basis and writes the data to  
Table xxx

In [93]:
import pyspark.sql.functions as sf
from time import time

Enriched Data

In [94]:
# Read data from 'runescape.02_silver.1h_prices_enriched'
df_prices = spark.read.table("runescape.02_silver.1h_prices_enriched")
df_prices = df_prices.drop("highalch","limit","members")

In [95]:
# get skill input data
df_skill_inputs = spark.read.table("runescape.02_silver.skill_action_inputs_enriched")

In [96]:
# Read data from 'runescape.02_silver.1h_prices_last_enriched'
df_prices_last = spark.read.table("runescape.02_silver.1h_prices_last_enriched")
# drop time column
df_prices_last = df_prices_last.drop("time", "highalch", "limit", "members")

In [97]:
# TODO replace the code below with code that gets the most recent data from 'runescape.02_silver.1h_prices_enriched'
# instead of using the time window
# will need to make sure that the merge works when writing to gold layer.

In [98]:
# filter data to only last 90 minutes (extra time allowed for any delays)
# job will run this notebook every hour
unix_time_window = time() - 8400 # 90 minutes
df_prices_latest = df_prices.filter(f"time > {unix_time_window}")
# validate there is only 1 value for time in df_prices_latest
time_range = df_prices_latest.select("time").distinct().count()
if time_range > 1:
    raise Exception("There is more than 1 time present in df_1h_prices_latest")
# df with the time for this data set.  This will be used later for the imputed data.
df_time = df_prices_latest.select(sf.first_value("time").alias("time"))
# check to make sure the time value is not null
if df_time.collect()[0][0] is None:
    raise Exception("The time in df_time is Null")

In [99]:
df_prices_latest.show()

+---+------------+------------------+-----------+-----------------+----------+--------------------+
| id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|      time|                name|
+---+------------+------------------+-----------+-----------------+----------+--------------------+
|  2|         305|            404808|        298|           428390|1772643600|    Steel cannonball|
| 36|         370|                29|        367|               11|1772643600|              Candle|
| 39|           5|              3769|          4|             2463|1772643600|    Bronze arrowtips|
| 40|           6|               180|          7|             2395|1772643600|      Iron arrowtips|
| 41|          18|              9927|         18|             2765|1772643600|     Steel arrowtips|
| 42|          40|             23128|         38|             5347|1772643600|   Mithril arrowtips|
| 43|         102|              6535|         98|            11190|1772643600|   Adamant arrowtips|


Need to create a dataframe for all skill action outputs (and prices/ volumes) and thier respective inputs (and prices/ volumes)  
This dataframe also should have price data (and inputs) for items that did not have data in the last hour and instead use historic data from table 1h_prices_last_enriched

In [100]:
# get df with all distinct skill action output and input id's
# cross join with df_time 
# TODO consider how this is impacted by implementation of other skills
# TODO consider renaming df_outputs_time to something less dumb
df_outputs_time = df_skill_inputs\
    .select("id_output")\
    .union(df_skill_inputs\
    .select("id_input"))\
    .dropDuplicates()\
    .withColumnRenamed("id_output","id")\
    .crossJoin(df_time)


Data imputation for missing/ filtered out data.  
Imputation method: Use the last known avg values based on hourly data in table 'runescape.02_silver.1h_prices_last_enriched'  
This approach has it's limiations and should be reconsidered.

In [101]:
# create df with imputed price data using the prices from 1h_prices_last_enriched
# This data will be used when there is no data for the respective item for profit calcs
df_prices_imputed = df_prices_last.join(df_outputs_time, "id")

In [102]:
# create df with only items that have missing 'real' price data
df_prices_missing = df_prices_imputed.join(df_prices_latest.select("id"),"id","left_anti")

In [103]:
df_prices_missing.sort("id").show()

+----+------------+------------------+-----------+-----------------+------------------+----------+
|  id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|              name|      time|
+----+------------+------------------+-----------+-----------------+------------------+----------+
|1095|           4|               488|          9|              147|     Leather chaps|1772643600|
|1133|         264|                26|        303|              186|      Studded body|1772643600|
|1641|        1156|                13|       1124|              710|         Ruby ring|1772643600|
|1664|       11789|               843|      11783|               41|   Dragon necklace|1772643600|
|1677|         686|             10000|        627|              321|Emerald amulet (u)|1772643600|
|1679|        1130|               573|       1106|               65|   Ruby amulet (u)|1772643600|
|1714|          19|                36|         20|              200|   Unstrung symbol|1772643600|
|1743|    

In [104]:
# union price data to get complete data set
df_prices_full = df_prices_missing.unionByName(df_prices_latest)

Build table with profit calculations

In [105]:
# create df with full price data set combined with inputs (qty, id's)
# this will be used to calculate the input cost for each skill action
# inner join to only get data that has skill action
# TODO make sure this shouldn't be a left_outer join
df_prices_full_w_inputs = df_prices_full.join(
    df_skill_inputs, df_prices_full.id == df_skill_inputs.id_output, "inner"
)
df_prices_full_w_inputs = df_prices_full_w_inputs.drop(
    "id", "name", "highalch", "limit", "members"
)
# rename columns to be associated with output data
df_prices_full_w_inputs = df_prices_full_w_inputs.withColumnsRenamed(
    {
        "avg1HourHigh": "output_avg_high",
        "avg1HourHighVolume": "output_high_volume",
        "avg1HourLow": "output_avg_low",
        "avg1HourLowVolume": "output_low_volume",
    }
)


In [106]:
df_prices_full_w_inputs.show()

+---------------+------------------+--------------+-----------------+----------+------------------+---------+------------+--------+---------+--------+
|output_avg_high|output_high_volume|output_avg_low|output_low_volume|      time|            output|input_qty|       input|   skill|id_output|id_input|
+---------------+------------------+--------------+-----------------+----------+------------------+---------+------------+--------+---------+--------+
|              4|               488|             9|              147|1772643600|     Leather chaps|      0.2|      Thread|crafting|     1095|    1734|
|            264|                26|           303|              186|1772643600|      Studded body|      1.0| Steel studs|crafting|     1133|    2370|
|          11789|               843|         11783|               41|1772643600|   Dragon necklace|      1.0|    Gold bar|crafting|     1664|    2357|
|            686|             10000|           627|              321|1772643600|Emerald amulet

In [ ]:
# create final price df with input costs
df_prices_final = df_prices_full_w_inputs.join(
    df_prices_full,
    [
        df_prices_full_w_inputs.id_input == df_prices_full.id,
        df_prices_full_w_inputs.time == df_prices_full.time,
    ],
    "left_outer",
).drop(df_prices_full_w_inputs.time, "id", "name")\
.withColumnsRenamed(
    {
        "avg1HourHigh": "input_avg_high",
        "avg1HourHighVolume": "input_high_volume",
        "avg1HourLow": "input_avg_low",
        "avg1HourLowVolume": "input_low_volume",
    }
)


In [108]:
df_prices_final.show()

+---------------+------------------+--------------+-----------------+------------------+---------+------------+--------+---------+--------+--------------+-----------------+-------------+----------------+----------+
|output_avg_high|output_high_volume|output_avg_low|output_low_volume|            output|input_qty|       input|   skill|id_output|id_input|input_avg_high|input_high_volume|input_avg_low|input_low_volume|      time|
+---------------+------------------+--------------+-----------------+------------------+---------+------------+--------+---------+--------+--------------+-----------------+-------------+----------------+----------+
|              4|               488|             9|              147|     Leather chaps|      0.2|      Thread|crafting|     1095|    1734|             7|             5718|            6|             781|1772643600|
|            264|                26|           303|              186|      Studded body|      1.0| Steel studs|crafting|     1133|    2370| 

In [109]:
# calculate input costs
df_input_cost = df_prices_final.withColumns(
    {
        "input_cost_low": sf.round(sf.col("input_avg_low") * sf.col("input_qty"), 1),
        "input_cost_high": sf.round(sf.col("input_avg_high") * sf.col("input_qty"), 1),
    }
)


In [ ]:
# Summarize total input costs for each output
df_input_total_cost = df_input_cost.groupBy("time", "output", "skill").sum(
    "input_cost_high", "input_cost_low"
)
df_input_total_cost = df_input_total_cost.withColumnsRenamed(
    {
        "sum(input_cost_high)": "total_input_cost_high",
        "sum(input_cost_low)": "total_input_cost_low",
    }
)


Calculate profit estimates based on input costs and GE Tax  
GE tax needs to be replaced with the GE tax function, but is close enough for now.

In [111]:
tax = 0.98  # simplified tax for now, can be implemented as function
df_profits = (
    df_prices_final.join(df_input_total_cost, ["time", "output", "skill"]).drop(
        "name"
    )  # TODO rename input cols
    # calculate profit estimates without tax
    # TODO add tax and implement as function
    .withColumns(
        {
            "profit_low_est": sf.round(
                tax * sf.col("output_avg_low") - sf.col("total_input_cost_high"), 2
            ),
            "profit_high_est": sf.round(
                tax * sf.col("output_avg_high") - sf.col("total_input_cost_low"), 2
            ),
        }
    )
)


In [112]:
df_profits.show()

+----------+--------------------+--------+---------------+------------------+--------------+-----------------+---------+--------------------+---------+--------+--------------+-----------------+-------------+----------------+---------------------+--------------------+--------------+---------------+
|      time|              output|   skill|output_avg_high|output_high_volume|output_avg_low|output_low_volume|input_qty|               input|id_output|id_input|input_avg_high|input_high_volume|input_avg_low|input_low_volume|total_input_cost_high|total_input_cost_low|profit_low_est|profit_high_est|
+----------+--------------------+--------+---------------+------------------+--------------+-----------------+---------+--------------------+---------+--------+--------------+-----------------+-------------+----------------+---------------------+--------------------+--------------+---------------+
|1772643600|     Air battlestaff|crafting|           8773|             34019|          8756|           